In [1]:
from src.data_processing.data_loader import MovieLensDataLoader
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
import pandas as pd

In [2]:


loader = MovieLensDataLoader()
data_dict = loader.load_data()
await loader.letterboxd_data_async(max_concurrent_requests=500)

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:9742 films...
INFO:src.data_processing.data_loader:Successfully fetched from API: 9623 films.


In [3]:
# 2. Подготовка DataFrame
movies_df = pd.DataFrame(loader.movie_data)
ratings_df = data_dict['ratings']
genre_features = loader.preprocess_movies()
movies_df = pd.concat([movies_df, genre_features], axis=1)
movies_df = movies_df.dropna().reset_index(drop=True)
movies_df.head(10)

,movieId,title,year,cast,main_actor,director,rating,runtime,keywords,vote_count,...,genre_film-noir,genre_horror,genre_imax,genre_musical,genre_mystery,genre_romance,genre_sci-fi,genre_thriller,genre_war,genre_western
0,1.0,Toy Story,1995,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",Tom Hanks,John Lasseter,7.978,81.0,"[rescue, friendship, mission, jealousy, villai...",20013.0,...,0,0,0,0,0,0,0,0,0,0
1,2.0,Jumanji,1995,"[Robin Williams, Kirsten Dunst, Bradley Pierce...",Robin Williams,Joe Johnston,7.247,104.0,"[based on novel or book, giant insect, board g...",11351.0,...,0,0,0,0,0,0,0,0,0,0
2,3.0,Grumpier Old Men,1995,"[Walter Matthau, Jack Lemmon, Ann-Margret, Sop...",Walter Matthau,Howard Deutch,6.478,101.0,"[fishing, sequel, old man, best friend, weddin...",431.0,...,0,0,0,0,0,1,0,0,0,0
3,4.0,Waiting to Exhale,1995,"[Whitney Houston, Angela Bassett, Loretta Devi...",Whitney Houston,Forest Whitaker,6.243,127.0,"[based on novel or book, single mother, divorc...",206.0,...,0,0,0,0,0,1,0,0,0,0
4,5.0,Father of the Bride Part II,1995,"[Steve Martin, Diane Keaton, Martin Short, Kim...",Steve Martin,Charles Shyer,6.300,106.0,"[daughter, baby, parent child relationship, mi...",811.0,...,0,0,0,0,0,0,0,0,0,0
5,6.0,Heat,1995,"[Al Pacino, Robert De Niro, Val Kilmer, Jon Vo...",Al Pacino,Michael Mann,7.937,170.0,"[robbery, chase, obsession, detective, heist, ...",8400.0,...,0,0,0,0,0,0,0,1,0,0
6,7.0,Sabrina,1995,"[Harrison Ford, Julia Ormond, Greg Kinnear, Na...",Harrison Ford,Sydney Pollack,6.212,127.0,"[chauffeur, sibling relationship, paris, franc...",695.0,...,0,0,0,0,0,1,0,0,0,0
7,8.0,Tom and Huck,1995,"[Jonathan Taylor Thomas, Brad Renfro, Eric Sch...",Jonathan Taylor Thomas,Peter Hewitt,5.300,97.0,"[based on novel or book, mississippi river, ma...",214.0,...,0,0,0,0,0,0,0,0,0,0
8,9.0,Sudden Death,1995,"[Jean-Claude Van Damme, Powers Boothe, Raymond...",Jean-Claude Van Damme,Peter Hyams,6.034,111.0,"[explosive, hostage, ice hockey, terrorism, vi...",812.0,...,0,0,0,0,0,0,0,0,0,0
9,10.0,GoldenEye,1995,"[Pierce Brosnan, Sean Bean, Izabella Scorupco,...",Pierce Brosnan,Martin Campbell,6.902,130.0,"[computer virus, cuba, falsely accused, secret...",4350.0,...,0,0,0,0,0,0,0,1,0,0


In [4]:
# 2. Конфигурация
config = ContentBasedConfig(
    main_actor_weight=0.3,
    director_weight=0.3,
    cast_weight=0.3,
    numerical_weight=0.1,
    similarity_threshold=0.15
)

# 3. Обучение модели (Передаем ОБА датафрейма!)
cb_model = ContentBasedRecommender(config=config)
# Обучение модели (как обычно)
cb_model.fit(movies_df=movies_df, ratings_df=ratings_df)

INFO:src.models.content_based:Starting fit process...
INFO:src.models.content_based:Step 1/5: Preprocessing numerical features...
INFO:src.models.content_based:Step 2/5: Preprocessing actor/director ratings...
INFO:src.models.content_based:Step 3/5: Building feature matrix...
INFO:src.models.content_based:Building feature matrix...
INFO:src.models.content_based:Combined features shape: (9589, 42245)
INFO:src.models.content_based:Non-zero elements: 294324
INFO:src.models.content_based:Step 4/5: Computing similarity matrix...
INFO:src.models.content_based:Computing cosine similarity matrix...
INFO:src.models.content_based:Similarity matrix shape: (9589, 9589)
INFO:src.models.content_based:Non-zero similarities: 367963
INFO:src.models.content_based:Step 5/5: Building user profiles from ratings...
INFO:src.models.content_based:Built sparse-optimized profiles for 609 users.
INFO:src.models.content_based:Fit process completed successfully!


In [8]:
user_id = 3

cb_model.show_user_profile_and_recommendations(
    user_id=user_id,
    ratings_df=ratings_df,
    movies_df=movies_df,
    k=10,
    top_rated_count=5,
    reasons_count=3
)

INFO:src.models.content_based:Generated 10 recommendations for user 3


USER PROFILE: 3

General Statistics:
   - Total ratings: 39
   - Average rating: 2.44 / 5.0
   - Unique movies rated: 39

TOP 5 FAVORITE MOVIES (Rating >= 4.0):
--------------------------------------------------------------------------------
 1. Escape from L.A.                                   [Rating: 5.0/5.0]
 2. Unknown (ID: 2851.0)                               [Rating: 5.0/5.0]
 3. The Lair of the White Worm                         [Rating: 5.0/5.0]
 4. Mad Max 2                                          [Rating: 5.0/5.0]
 5. Hangar 18                                          [Rating: 5.0/5.0]

PERSONALIZED RECOMMENDATIONS FOR USER 3

 1. Escape from New York
    Predicted Relevance Score: 0.3204
    Recommended because you liked:
       - Escape from L.A. (You rated: 5.0/5.0) [Similarity: 0.6744]
       - The Thing (You rated: 4.0/5.0) [Similarity: 0.4069]
       - Big Trouble in Little China [Similarity: 0.3606]

 2. Mad Max Beyond Thunderdome
    Predicted Relevance Score: 0.2